<a href="https://colab.research.google.com/github/digitalirrigation-lgtm/kefa-carbon-verification/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap

In [ ]:
import ee
import geemap
ee.Authenticate()

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

In [ ]:
print(ee.Number(1).add(1).getInfo())

In [ ]:
import ee
import geemap

ee.Initialize(project='irrigation-intelligence-pro')

# Load GAUL 2025 level 2 (zone level)
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')

# Filter by country first
ethiopia = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))

# Check what regions exist in Ethiopia
regions = ethiopia.aggregate_array('GAUL1_NAME').distinct().getInfo()
print('Regions in Ethiopia:')
for r in sorted(regions):
    print('  -', r)

In [ ]:
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
ethiopia = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))

# Find every row whose zone name contains "Kafa"
kafa_rows = ethiopia.filter(ee.Filter.stringContains('GAUL2_NAME', 'Kafa'))

# Print the region name and zone name for those rows
info = kafa_rows.aggregate_array('GAUL1_NAME').distinct().getInfo()
zones = kafa_rows.aggregate_array('GAUL2_NAME').distinct().getInfo()

print('Region(s) that contain Kafa:')
for r in info:
    print('  -', r)

print('Zone name(s) that match "Kafa":')
for z in zones:
    print('  -', z)

In [ ]:
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')

# Filter: country = Ethiopia, region = South West Ethiopia
sw = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
sw = sw.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))

# Get every zone name in that region
zones = sw.aggregate_array('GAUL2_NAME').distinct().sort().getInfo()

print('Number of zones:', len(zones))
print('Zone names in South West Ethiopia:')
for z in zones:
    print('  -', z)

In [ ]:
import ee
import geemap

ee.Initialize(project='irrigation-intelligence-pro')

# Load GAUL 2025 level 2
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')

# Filter to Kefa zone
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))

# Turn it into one pure geometry
aoi = kefa.geometry()

# Show on a map
m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(aoi, {'color': 'red'}, 'Kefa Zone')
m

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# Rebuild Kefa boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# Load NASA/ORNL biomass carbon dataset
carbon_collection = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1')

# Filter to Kefa only
carbon_kefa = carbon_collection.filterBounds(aoi)

# There is only one image (2010). Take it.
carbon_image = carbon_kefa.first()

# What bands are inside this image?
band_names = carbon_image.bandNames().getInfo()
print('Bands in the NASA carbon image:')
for b in band_names:
    print('  -', b)

# What is the native scale?
scale = carbon_image.projection().nominalScale().getInfo()
print('Native scale (meters):', scale)

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# Rebuild Kefa boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# Load NASA 2010 carbon image for Kefa
carbon_image = (
    ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1')
    .filterBounds(aoi)
    .first()
)
agb = carbon_image.select('agb')   # aboveground carbon, Mg C/ha

# Convert each pixel from Mg C/ha to tonnes C, then sum
carbon_per_pixel = agb.multiply(ee.Image.pixelArea()).divide(10000)

total = carbon_per_pixel.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=309,          # native scale of the NASA dataset
    crs='EPSG:32637',   # UTM 37N, your chosen CRS
    maxPixels=1e13
)

total_tC = total.get('agb').getInfo()
print('Total aboveground carbon in Kefa Zone (tC):', total_tC)
print('Total CO2-equivalent (tCO2e):', total_tC * 44 / 12)

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

carbon_image = (
    ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1')
    .filterBounds(aoi)
    .first()
)

agb = carbon_image.select('agb')
unc = carbon_image.select('agb_uncertainty')

# Carbon total (same as before)
carbon_total = agb.multiply(ee.Image.pixelArea()).divide(10000).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi,
    scale=309, crs='EPSG:32637', maxPixels=1e13
).get('agb').getInfo()

# Uncertainty total
unc_total = unc.multiply(ee.Image.pixelArea()).divide(10000).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi,
    scale=309, crs='EPSG:32637', maxPixels=1e13
).get('agb_uncertainty').getInfo()

print('Central estimate (tC):', carbon_total)
print('Uncertainty (tC, 1 sigma):', unc_total)
print('Relative uncertainty (%):', 100 * unc_total / carbon_total)
print('Plausible range (tC):', carbon_total - unc_total, 'to', carbon_total + unc_total)

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

carbon_image = (
    ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1')
    .filterBounds(aoi)
    .first()
)
agb = carbon_image.select('agb').clip(aoi)

m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(agb, {
    'min': 0, 'max': 150,
    'palette': ['white', 'yellow', 'orange', 'red', 'darkred']
}, 'Aboveground carbon (Mg C/ha)')
m.addLayer(aoi, {'color': 'black'}, 'Kefa boundary')
m

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Get the Kefa boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# 2. Load the carbon image and clip it to Kefa
carbon = (
    ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1')
    .filterBounds(aoi).first()
)
agb = carbon.select('agb').clip(aoi)

# 3. Create a map with a blank background
m = geemap.Map()
m.addLayer(agb, {
    'min': 0, 'max': 150,
    'palette': ['white', 'yellow', 'orange', 'red', 'darkred']
}, 'Aboveground carbon (Mg C/ha)')
m.addLayer(aoi, {'color': 'black'}, 'Kefa boundary')
m.centerObject(aoi, 9)

# 4. This line removes the background map completely
m.remove(m.layers[0])

m

In [ ]:
m2 = geemap.Map()
m2.centerObject(aoi, 9)
m2.addLayer(agb, {'min': 0, 'max': 150, 'palette': palette}, 'Aboveground carbon')
m2.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')
m2.add_colorbar(
    agb,
    colors=palette,
    vmin=0,
    vmax=150,
    caption='Aboveground carbon (Mg C/ha)',
    position='bottomright'
)
m2

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# --- Kefa boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon image ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# --- BOLD palette defined inside this cell ---
palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Map with continuous colour bar ---
m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(agb, {'min': 0, 'max': 150, 'palette': palette}, 'Aboveground carbon')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')

m.add_colorbar(
    agb,
    colors=palette,
    vmin=0,
    vmax=150,
    caption='Aboveground carbon (Mg C/ha)',
    position='bottomright'
)

m

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(agb, {'min': 0, 'max': 150, 'palette': palette}, 'Aboveground carbon')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')

legend = {
    '0 – 25 Mg C/ha  (bare, water)': '#D9D9D9',
    '25 – 50  (farmland, grass)': '#FFFF00',
    '50 – 75  (shrub, mixed)': '#FFA500',
    '75 – 100  (woodland)': '#FF4500',
    '100 – 125  (secondary forest)': '#FF0000',
    '125 – 150  (mature forest)': '#8B0000',
    '150+  (very dense forest)': '#000000'
}
m.add_legend(title='Aboveground carbon (Mg C/ha)', legend_dict=legend, position='bottomright')
m

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# --- Kefa boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon image ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Create map and REMOVE the default background layer ---
m = geemap.Map()
m.remove(m.layers[0])          # <-- this removes the blocked Google tiles

# --- Add ONLY your layers ---
m.centerObject(aoi, 9)
m.addLayer(agb, {'min': 0, 'max': 150, 'palette': palette}, 'Aboveground carbon')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')

# --- Legend ---
legend = {
    '0 – 25 Mg C/ha  (bare, water)': '#D9D9D9',
    '25 – 50  (farmland, grass)': '#FFFF00',
    '50 – 75  (shrub, mixed)': '#FFA500',
    '75 – 100  (woodland)': '#FF4500',
    '100 – 125  (secondary forest)': '#FF0000',
    '125 – 150  (mature forest)': '#8B0000',
    '150+  (very dense forest)': '#000000'
}
m.add_legend(title='Aboveground carbon (Mg C/ha)', legend_dict=legend, position='bottomright')

m

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# --- Kefa boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon image ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# --- Classify continuous carbon into 7 discrete bins ---
classified = agb.multiply(0).toInt()            # zeros on agb's grid, mask preserved
classified = classified.where(agb.gte(25),  1)
classified = classified.where(agb.gte(50),  2)
classified = classified.where(agb.gte(75),  3)
classified = classified.where(agb.gte(100), 4)
classified = classified.where(agb.gte(125), 5)
classified = classified.where(agb.gte(150), 6)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Map ---
m = geemap.Map()
m.remove(m.layers[0])
m.centerObject(aoi, 9)
m.addLayer(classified, {'min': 0, 'max': 6, 'palette': palette}, 'Carbon class')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')

# --- Legend (same 7 hex codes, same order) ---
legend = {
    '0 – 25 Mg C/ha  (bare, water)':       '#D9D9D9',
    '25 – 50  (farmland, grass)':          '#FFFF00',
    '50 – 75  (shrub, mixed)':             '#FFA500',
    '75 – 100  (woodland)':                '#FF4500',
    '100 – 125  (secondary forest)':       '#FF0000',
    '125 – 150  (mature forest)':          '#8B0000',
    '150+  (very dense forest)':           '#000000'
}
m.add_legend(title='Aboveground carbon (Mg C/ha)', legend_dict=legend, position='bottomright')
m

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# Kefa boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# Carbon image
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# Classify to integers 0–6 (exact bins, exact colours, no interpolation)
classified = agb.multiply(0).toInt()
classified = classified.where(agb.gte(25),  1)
classified = classified.where(agb.gte(50),  2)
classified = classified.where(agb.gte(75),  3)
classified = classified.where(agb.gte(100), 4)
classified = classified.where(agb.gte(125), 5)
classified = classified.where(agb.gte(150), 6)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

m = geemap.Map()
m.remove(m.layers[0])
m.centerObject(aoi, 9)
m.addLayer(classified, {'min': 0, 'max': 6, 'palette': palette}, 'Carbon class')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa boundary')
m

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']
labels = [
    '0 – 25  Mg C/ha  (bare land, water)',
    '25 – 50  (farmland, grassland)',
    '50 – 75  (shrub, mixed cover)',
    '75 – 100  (woodland)',
    '100 – 125  (secondary forest)',
    '125 – 150  (mature forest)',
    '150+  (very dense forest)'
]

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(labels) + 1)
ax.axis('off')
ax.set_title('Aboveground carbon (Mg C/ha)', fontsize=12, loc='left', fontweight='bold')

for i, (label, color) in enumerate(zip(labels, palette)):
    y = len(labels) - i
    rect = mpatches.Rectangle((0.02, y - 0.4), 0.10, 0.7,
                              facecolor=color, edgecolor='black', linewidth=0.8)
    ax.add_patch(rect)
    ax.text(0.15, y - 0.05, label, fontsize=9, va='center')

plt.tight_layout()
plt.savefig('carbon_legend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# --- Kefa boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon image ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# --- Classify ---
classified = agb.multiply(0).toInt()
classified = classified.where(agb.gte(25),  1)
classified = classified.where(agb.gte(50),  2)
classified = classified.where(agb.gte(75),  3)
classified = classified.where(agb.gte(100), 4)
classified = classified.where(agb.gte(125), 5)
classified = classified.where(agb.gte(150), 6)

# --- FILL masked pixels with class 0 (grey) so no basemap shows through ---
classified = classified.unmask(0).clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Map with WHITE background (no green tiles) ---
m = geemap.Map(basemap='CartoDB.PositronNoLabels')
m.centerObject(aoi, 9)
m.addLayer(classified, {'min': 0, 'max': 6, 'palette': palette}, 'Aboveground carbon')
m.addLayer(aoi, {'color': '#00FFFF', 'fillColor': '00000000'}, 'Kefa boundary')

# --- Legend ON the map, bottom right ---
legend = {
    '0 – 25 Mg C/ha  (bare, water)':  '#D9D9D9',
    '25 – 50  (farmland, grass)':     '#FFFF00',
    '50 – 75  (shrub, mixed)':        '#FFA500',
    '75 – 100  (woodland)':           '#FF4500',
    '100 – 125  (secondary forest)':  '#FF0000',
    '125 – 150  (mature forest)':     '#8B0000',
    '150+  (very dense forest)':      '#000000'
}
m.add_legend(title='Aboveground carbon (Mg C/ha)', legend_dict=legend, position='bottomright')
m

In [ ]:
---------------------------------------------------------------------------
TraitError                                Traceback (most recent call last)
/tmp/ipykernel_10915/2630493404.py in <cell line: 0>()
     28
     29 # --- Map with WHITE background (no green tiles) ---
---> 30 m = geemap.Map(basemap='CartoDB.PositronNoLabels')
     31 m.centerObject(aoi, 9)
     32 m.addLayer(classified, {'min': 0, 'max': 6, 'palette': palette}, 'Aboveground carbon')

9 frames
/usr/local/lib/python3.13/dist-packages/traitlets/traitlets.py in error(self, obj, value, error, info)
    839                         describe("the", value),
    840                     )
--> 841                 raise TraitError(e)
    842
    843     def get_metadata(self, key, default=None):

TraitError: The 'basemap' trait of a Map instance expected a dict or a TileProvider or a TileLayer, not the str 'CartoDB.PositronNoLabels'.

In [ ]:
---------------------------------------------------------------------------
TraitError                                Traceback (most recent call last)
/tmp/ipykernel_10915/2630493404.py in <cell line: 0>()
     28
     29 # --- Map with WHITE background (no green tiles) ---
---> 30 m = geemap.Map(basemap='CartoDB.PositronNoLabels')
     31 m.centerObject(aoi, 9)
     32 m.addLayer(classified, {'min': 0, 'max': 6, 'palette': palette}, 'Aboveground carbon')

9 frames
/usr/local/lib/python3.13/dist-packages/traitlets/traitlets.py in error(self, obj, value, error, info)
    839                         describe("the", value),
    840                     )
--> 841                 raise TraitError(e)
    842
    843     def get_metadata(self, key, default=None):

TraitError: The 'basemap' trait of a Map instance expected a dict or a TileProvider or a TileLayer, not the str 'CartoDB.PositronNoLabels'.

In [ ]:
import ee, geemap, folium
ee.Initialize(project='irrigation-intelligence-pro')

# Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# Carbon
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# Classify to 7 bins, fill holes
c = agb.multiply(0).toInt()
c = c.where(agb.gte(25),  1)
c = c.where(agb.gte(50),  2)
c = c.where(agb.gte(75),  3)
c = c.where(agb.gte(100), 4)
c = c.where(agb.gte(125), 5)
c = c.where(agb.gte(150), 6)
c = c.unmask(0).clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# Map — kill basemap, add layers
m = geemap.Map()
m.remove(m.layers[0])
m.centerObject(aoi, 9)
m.addLayer(c, {'min': 0, 'max': 6, 'palette': palette, 'opacity': 1.0}, 'Carbon')
m.addLayer(aoi, {'color': '#00FFFF'}, 'Kefa')

# HTML legend — exact same hex codes as palette, on the SAME map
legend_html = '''
<div style="position:fixed;bottom:30px;right:30px;background:white;border:2px solid #333;padding:12px;font-size:12px;font-family:sans-serif;z-index:9999;">
<b style="font-size:13px;">Aboveground carbon (Mg C/ha)</b><br><br>
<div><span style="display:inline-block;width:22px;height:14px;background:#D9D9D9;border:1px solid #666;"></span>  0 – 25  bare, water</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFFF00;border:1px solid #666;"></span>  25 – 50  farmland, grass</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFA500;border:1px solid #666;"></span>  50 – 75  shrub, mixed</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF4500;border:1px solid #666;"></span>  75 – 100  woodland</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF0000;border:1px solid #666;"></span>  100 – 125  secondary forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#8B0000;border:1px solid #666;"></span>  125 – 150  mature forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#000000;border:1px solid #666;"></span>  150+  very dense forest</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
m

In [ ]:
import ee, folium
from folium import raster_layers

ee.Initialize(project='irrigation-intelligence-pro')

# --- Boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# --- Classify to 7 bins, fill holes ---
c = agb.multiply(0).toInt()
c = c.where(agb.gte(25),  1)
c = c.where(agb.gte(50),  2)
c = c.where(agb.gte(75),  3)
c = c.where(agb.gte(100), 4)
c = c.where(agb.gte(125), 5)
c = c.where(agb.gte(150), 6)
c = c.unmask(0).clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Get tile URLs from Earth Engine ---
carbon_mapid  = c.getMapId({'min': 0, 'max': 6, 'palette': palette, 'opacity': 1.0})
boundary_mapid = aoi.getMapId({'color': '#00FFFF', 'width': 2})

# --- folium map with NO basemap (tiles=None = pure white background) ---
m = folium.Map(location=[7.3, 36.2], zoom_start=9, tiles=None)

raster_layers.TileLayer(
    tiles=carbon_mapid['tile_fetcher'].url_format,
    attr='GEE',
    name='Aboveground carbon',
    overlay=True,
    show=True,
).add_to(m)

raster_layers.TileLayer(
    tiles=boundary_mapid['tile_fetcher'].url_format,
    attr='GEE',
    name='Kefa boundary',
    overlay=True,
    show=True,
).add_to(m)

# --- HTML legend, exact same hex codes as palette ---
legend_html = '''
<div style="position:fixed;bottom:30px;right:30px;background:white;border:2px solid #333;padding:12px;font-size:12px;font-family:sans-serif;z-index:9999;box-shadow:0 0 6px rgba(0,0,0,0.3);">
<b>Aboveground carbon (Mg C/ha)</b><br><br>
<div><span style="display:inline-block;width:22px;height:14px;background:#D9D9D9;border:1px solid #666;"></span>&nbsp;0 &ndash; 25 &nbsp; bare, water</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFFF00;border:1px solid #666;"></span>&nbsp;25 &ndash; 50 &nbsp; farmland, grass</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFA500;border:1px solid #666;"></span>&nbsp;50 &ndash; 75 &nbsp; shrub, mixed</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF4500;border:1px solid #666;"></span>&nbsp;75 &ndash; 100 &nbsp; woodland</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF0000;border:1px solid #666;"></span>&nbsp;100 &ndash; 125 &nbsp; secondary forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#8B0000;border:1px solid #666;"></span>&nbsp;125 &ndash; 150 &nbsp; mature forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#000000;border:1px solid #666;"></span>&nbsp;150+ &nbsp; very dense forest</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

In [ ]:
import ee, folium
from folium import raster_layers

ee.Initialize(project='irrigation-intelligence-pro')

# --- Boundary ---
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
kefa = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL1_NAME', 'South West Ethiopia'))
kefa = kefa.filter(ee.Filter.eq('GAUL2_NAME', 'Kefa'))
aoi = kefa.geometry()

# --- Carbon ---
carbon = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').filterBounds(aoi).first()
agb = carbon.select('agb').clip(aoi)

# --- Classify to 7 bins, fill holes ---
c = agb.multiply(0).toInt()
c = c.where(agb.gte(25),  1)
c = c.where(agb.gte(50),  2)
c = c.where(agb.gte(75),  3)
c = c.where(agb.gte(100), 4)
c = c.where(agb.gte(125), 5)
c = c.where(agb.gte(150), 6)
c = c.unmask(0).clip(aoi)

palette = ['#D9D9D9', '#FFFF00', '#FFA500', '#FF4500', '#FF0000', '#8B0000', '#000000']

# --- Boundary as an IMAGE (this is the fix) ---
boundary_img = ee.Image().byte().paint(aoi, 1, 2)

# --- Get tile URLs ---
carbon_mapid   = c.getMapId({'min': 0, 'max': 6, 'palette': palette, 'opacity': 1.0})
boundary_mapid = boundary_img.getMapId({'palette': ['#00FFFF']})

# --- folium map, no basemap ---
m = folium.Map(location=[7.3, 36.2], zoom_start=9, tiles=None)

raster_layers.TileLayer(
    tiles=carbon_mapid['tile_fetcher'].url_format,
    attr='GEE', name='Aboveground carbon', overlay=True, show=True,
).add_to(m)

raster_layers.TileLayer(
    tiles=boundary_mapid['tile_fetcher'].url_format,
    attr='GEE', name='Kefa boundary', overlay=True, show=True,
).add_to(m)

# --- Legend, exact same hex codes as palette ---
legend_html = '''
<div style="position:fixed;bottom:30px;right:30px;background:white;border:2px solid #333;padding:12px;font-size:12px;font-family:sans-serif;z-index:9999;box-shadow:0 0 6px rgba(0,0,0,0.3);">
<b>Aboveground carbon (Mg C/ha)</b><br><br>
<div><span style="display:inline-block;width:22px;height:14px;background:#D9D9D9;border:1px solid #666;"></span>&nbsp;0 &ndash; 25 &nbsp; bare, water</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFFF00;border:1px solid #666;"></span>&nbsp;25 &ndash; 50 &nbsp; farmland, grass</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FFA500;border:1px solid #666;"></span>&nbsp;50 &ndash; 75 &nbsp; shrub, mixed</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF4500;border:1px solid #666;"></span>&nbsp;75 &ndash; 100 &nbsp; woodland</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#FF0000;border:1px solid #666;"></span>&nbsp;100 &ndash; 125 &nbsp; secondary forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#8B0000;border:1px solid #666;"></span>&nbsp;125 &ndash; 150 &nbsp; mature forest</div>
<div><span style="display:inline-block;width:22px;height:14px;background:#000000;border:1px solid #666;"></span>&nbsp;150+ &nbsp; very dense forest</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m